# Module 04 — Functions

You have written functions for four semesters. What follows is the list of things
Python does differently, and one of them is a trap that catches everybody once.

## 1. `def`, and what is missing from it

No return type in the signature, no access modifier, no `throws`. A function that
falls off the end returns `None` — there is no `void`, only a value nobody looks at.

In [ ]:
def classify(reading: float) -> str:
    """Return the state of a reading."""
    if reading > 85:
        return "above limit"
    return "plausible"


print(classify(91.0))
print(classify.__doc__)

The annotations are notation, not enforcement — module 00 showed `mypy` doing the
checking. The docstring is the first statement in the body and is a real attribute
afterwards, which is what `help()` reads.

## 2. There is no overloading

A `def` binds a name, exactly like `=` does. A second `def` with the same name
rebinds it and the first one is gone — signature and all.

`ruff` flags the second one as `F811`, "redefinition of unused name". The
`# noqa: F811` comment on that line switches the rule off for that line — here the
redefinition is the point, so the linter is overruled on purpose.

In [ ]:
def area(side):
    return side * side


def area(width, height):  # noqa: F811 -- rebinding the name is the point
    return width * height


# What does calling it with one argument do now?
try:
    area(3)
    outcome = "worked"
except TypeError:
    outcome = "TypeError"

assert outcome == ...

Java would pick the right overload by signature. Python has one name and one
function, so the same job is done with default arguments, `*args`, or a check
inside the body. That is not a workaround — it is why the argument machinery in
the next sections is as rich as it is.

## 3. Returning more than one value

`return a, b` builds a tuple, and the caller takes it apart on assignment. No
out-parameters, no wrapper class.

In [ ]:
def summarise(readings):
    return min(readings), max(readings), sum(readings) / len(readings)


low, high, mean = summarise([21.7, 23.1, 22.4])
print(low, high, round(mean, 2))
print(type(summarise([1.0])))

## 4. Default arguments are evaluated once

This is the trap. The default expression runs **when the `def` runs**, not on each
call — so a mutable default is one object shared by every call that omits it.

In [ ]:
def collect(item, bucket=[]):
    bucket.append(item)
    return bucket


collect(1)
collect(2)

# What comes back from the third call?
assert collect(3) == ...

The proof is that the default is a normal attribute of the function object, and
you can watch it grow:

In [ ]:
def collect(item, bucket=[]):
    bucket.append(item)
    return bucket


print(collect.__defaults__)
collect(1)
collect(2)
print(collect.__defaults__)

The fix is always the same: default to `None` and build the real value inside.

```python
def collect(item, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(item)
    return bucket
```

C++ re-evaluates a default on every call, Java has no defaults at all — so this
is a rule with no equivalent to fall back on. Immutable defaults (`0`, `""`,
`None`, a tuple) are safe, which is why the trap stays hidden until the day
somebody writes `= []`.

## 5. Arguments: positional, keyword, `*args`, `**kwargs`

Any parameter can be passed by name at the call site, which makes a call
self-documenting without a comment. A `*` in the signature marks everything after
it as **keyword-only**, so you can forbid the unreadable call.

In [ ]:
def log(tag, unit="C", *extra, precision=1, **meta):
    return tag, unit, extra, precision, meta


print(log("TH-04"))
print(log("TH-04", "F", "spare", precision=2, location="hall 2"))

In [ ]:
def scale(value, *, factor):  # factor cannot be passed positionally
    return value * factor


print(scale(2, factor=3))
scale(2, 3)

`*extra` collects surplus positional arguments into a tuple, `**meta` surplus
keyword arguments into a dict. The same two stars work at the call site to unpack:
`log(*names)` and `log(**settings)`.

## 6. What the caller sees

Arguments are passed as references to objects. **Mutating** the object is visible
to the caller; **rebinding** the parameter is not.

In [ ]:
def mutate(values):
    values.append(99)


def rebind(values):
    values = [0]  # noqa: F841 -- the unused rebinding is the point


data = [1]
mutate(data)
rebind(data)

assert data == ...

`mutate` reached the caller's list; `rebind` only moved its own local name. This
is the same rule as `b = a` in module 01, seen from the other side — and module 05
is where it stops being a curiosity.

## 7. Scope, and `nonlocal`

A name assigned anywhere in a function is local to that whole function — there is
no block scope, and no declaration point. To assign to a name from an enclosing
function, say `nonlocal`; for a module-level one, `global`.

In [ ]:
def counter():
    count = 0

    def tick():
        nonlocal count  # without this, count = count + 1 is a local
        count += 1

    tick()
    tick()
    return count


print(counter())

A closure captures the **variable**, not the value it held when the closure was
built. That is Java's effectively-final restriction removed: it is what lets
`counter` accumulate without a wrapper object — and what makes closures built in a
loop share one variable.

In [ ]:
def make_handlers():
    handlers = []
    for i in range(3):

        def show():
            print(i)  # i is the enclosing variable, not a copy

        handlers.append(show)
    return handlers


for h in make_handlers():
    h()  # 2, 2, 2 -- all three share the one i

All three print `2`, because all three read the same `i` — and by the time they
run, the loop has finished with it at 2. Binding the value at creation time is the
usual fix: `def show(i=i):`.

Java forbids this by construction. Python trades that safety for the accumulator
above.

## 8. Functions are values

A function is an object: assign it, pass it, put it in a list. The everyday use is
`key=`, which takes the function to sort by.

In [ ]:
readings = [("TH-04", 91.0), ("TH-01", 21.7), ("TH-09", 23.1)]


def by_value(entry):
    return entry[1]


print(sorted(readings, key=by_value))
print(sorted(readings, key=by_value, reverse=True)[0])

# lambda is the same thing without a name, for one-liners
print(sorted(readings, key=lambda entry: entry[0]))

Java needs a functional interface and a method reference for this; here the
function itself is the value. Decorators, in module 14, are the next step from
here.

---

On to `exercises/`, then `uv run pytest 04_functions`.